# 02A Full Refresh Demo

Run the completed retail example: **Full Read → PySpark transformation → Overwrite**. This notebook preserves the working Orders, Products, and Order History pipeline that previously lived in `02_pipeline.ipynb`.

# 0. Environment — Setup Notebook

Run `setup_notebook()` to load `00_env_config`, resolve the caller environment, validate targets, and display configuration readiness. For shared configuration, pass `config_workspace_id` here. Setup Pipeline reuses this configuration.

In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import finalize_pipeline, orchestrate_read, orchestrate_write, setup_notebook, setup_pipeline

context = setup_notebook()

# For shared config, use the workspace hosting 00_env_config:
# context = setup_notebook(
#     config_workspace_id="<configuration-workspace-id>",
# )


# 1. Setup Pipeline

Use **Dry Run** to preview without writing, **Ungoverned** to run without Data Contracts, or **Governed** to enforce activated Data Contracts. Pipeline source participation and reporting are automatic.

Run the final Finalize cell after the complete pipeline. Dry Run returns a session-only summary; persistent modes record the actual outcome.


In [ ]:
# Ungoverned automatically registers or recovers this notebook's permanent UUID.
# Retain an explicit UUID in source only for cross-workspace promotion.
PIPELINE_ID = None
PIPELINE_MODE = "Ungoverned"  # Dry Run, Ungoverned, Governed

pipeline = setup_pipeline(
    pipeline_id=PIPELINE_ID,
    pipeline_mode=PIPELINE_MODE,
)
PIPELINE_ID = pipeline.pipeline_id


# 2. Read

`orchestrate_read()` is the plug-and-play governed source read. You provide the source location and read behaviour; FabricOps runs the standard source-side lifecycle underneath and returns the DataFrame plus the results of each check.

**Under the hood**

- **Read** — read the governed source.
- **Freshness** — check the source freshness expectation.
- **Schema** — validate the observed schema against the Data Contract.
- **Grain** — evaluate the configured Grain Guardrail.
- **Data Quality** — evaluate configured DQ rules.
- **Profile** — profile the full source for governance and inspection.

**Parameters**

- The `sources` dictionary key — notebook-facing label, not an orchestrator argument.
- `store` — configured FabricStore key from `00_env_config` such as `Bronze`, `Silver`, or `Gold`.
- `schema` — physical Lakehouse or Warehouse schema containing the source table.
- `table_name` — physical source table name.
- `read_mode` — `full` for the complete source or `incremental` for target-aware incremental processing.
- `query` — optional read-only Warehouse SQL pushdown. Use `None` for a normal table read.
- `read_parameters` — Incremental options such as `watermark_column`; not needed for these Full reads.
- `target_table_id` — required for Incremental reads so FabricOps can resolve target-aware state.

The returned object includes `dataframe`, `table_id`, source check results, profiling results, and orchestration stage metadata. Optional inspection lines are kept commented out so normal runs stay clean.

Clone a Read block for each governed source.

In [ ]:
# Keep complete Read results for transformation and lineage.
sources = {}


## READ 1 — Orders

In [ ]:
source = orchestrate_read(

    store="Bronze",
    schema="demo",
    table_name="orders",
    read_mode="full",
    query=None,

    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)
sources["orders"] = source
# Inspect the actual rows returned by this Read.
display(sources["orders"]["dataframe"].limit(10))


## READ 2 — Products

In [ ]:
source = orchestrate_read(

    store="Bronze",
    schema="demo",
    table_name="products",
    read_mode="full",
    query=None,

    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)
sources["products"] = source
# Inspect the actual rows returned by this Read.
display(sources["products"]["dataframe"].limit(10))


## READ 3 — Order History

In [ ]:
source = orchestrate_read(

    store="Gold",
    schema="demo",
    table_name="order_history",
    read_mode="full",
    query="""
SELECT
    historical_order_id,
    customer_id,
    order_datetime,
    net_amount
FROM demo.order_history
""",

    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)
sources["history"] = source
# Inspect the actual rows returned by this Read.
display(sources["history"]["dataframe"].limit(10))


# 3. Transform

Use the named source DataFrames to build the DataFrames you want to publish. This example produces two different outputs so the WRITE blocks demonstrate publishing different tables.

In [ ]:
# Pull the source DataFrames back out of the sources dictionary by name.
orders_df = sources["orders"]["dataframe"]
products_df = sources["products"]["dataframe"]
history_df = sources["history"]["dataframe"]

history_summary_df = (
    history_df
    .groupBy("customer_id")
    .agg(
        F.count("*").alias("historical_order_count"),
        F.sum("net_amount").alias("historical_net_amount"),
        F.max("order_datetime").alias("latest_historical_order_datetime"),
    )
)

# Output 1: detailed order-level table.
transformed_df = (
    orders_df.alias("orders")
    .join(products_df.alias("products"), on="product_id", how="left")
    .join(history_summary_df.alias("history"), on="customer_id", how="left")
    .withColumn("order_net_amount", F.round(F.col("quantity") * F.col("unit_price") * (F.lit(1.0) - F.col("discount")), 2))
    .fillna({"historical_order_count": 0, "historical_net_amount": 0.0})
    .select(
        "order_id", "customer_id", "order_datetime", "modified_datetime",
        "product_id", "product_name", "product_category", "quantity", "unit_price",
        "discount", "order_net_amount", "order_status", "shipping_country",
        "historical_order_count", "historical_net_amount", "latest_historical_order_datetime",
    )
)

# Output 2: customer-level summary derived from the detailed order output.
customer_summary_df = (
    transformed_df
    .groupBy("customer_id")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.sum("order_net_amount").alias("total_order_net_amount"),
        F.max("order_datetime").alias("latest_order_datetime"),
        F.max("historical_order_count").alias("historical_order_count"),
        F.max("historical_net_amount").alias("historical_net_amount"),
    )
)

# Optional development inspection — uncomment only when needed.
# Detailed order-level transformed output that will feed WRITE 1.
# display(transformed_df)
# Customer-level aggregated output that will feed WRITE 2.
# display(customer_summary_df)


# 4. Write

`orchestrate_write()` is the shared target write. You pass the transformed DataFrame, destination, and load strategy; FabricOps applies the selected mode, mandatory safety checks, and any Governed contract guardrails before publication.

**Under the hood**

- **Schema** — validate the target schema against the Data Contract.
- **Sensitive Data** — detect and apply configured sensitive-data treatment.
- **Source Drift** — check governed source changes against target expectations.
- **Grain** — evaluate the configured target Grain Guardrail.
- **Data Quality** — evaluate configured DQ rules.
- **Write** — publish the target in Ungoverned or Governed mode.
- **Profile** — profile the persisted target after a successful write.

In **Dry Run**, every declared Write evaluates its prepared input and returns diagnostics without target publication or operational metadata commits. In **Ungoverned** and **Governed**, successful checks continue through publication and persisted-target profiling. Inspect each target result; an unpublished result does not imply its checks passed.

**Parameters**

- first argument — transformed Spark DataFrame to validate and publish.
- The `writes` dictionary key — notebook-facing label, not an orchestrator argument.
- `store` — configured destination FabricStore key from `00_env_config`.
- `schema` — physical target schema.
- `table_name` — physical target table name.
- `write_mode` — governed write strategy such as `overwrite`, `append`, `scd1`, or `scd2`.
- `write_parameters` — mode-specific options, such as SCD keys and effective dates.
- `repartition_by` — optional Spark partition count before the write.
- `existing_mapping` — optional caller-managed Sensitive Data token mapping; omit when a managed mapping store is configured.

The returned object includes the target `table_id`, guardrail results, publication status, orchestration stage metadata, and, when published, persisted-target profiling results. Optional inspection lines are kept commented out so normal runs stay clean.

Clone a Write block for each governed target.

In [ ]:
# Keep complete Write results for optional inspection.
writes = {}


## WRITE 1 — Lakehouse Curated Orders

In [ ]:
write_result = orchestrate_write(
    transformed_df,

    store="Silver",
    schema="demo",
    table_name="curated_orders",
    write_mode="overwrite",

    repartition_by=None,

    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)
writes["curated_orders_lakehouse"] = write_result
# Inspect the DataFrame submitted to this Write.
display(write_result["dataframe"].limit(10))

# Optional: inspect Data Quality diagnostics and caller-managed token mapping.
# display(write_result.get("dq_result"))
# display((write_result.get("sensitive_result") or {}).get("support_mapping"))

# Optional: read the persisted target after a successful persistent execution.
# from fabricops_kit import read_lakehouse_table
# display(read_lakehouse_table(store="Silver", schema="demo", table_name="curated_orders").limit(10))


## WRITE 2 — Warehouse Customer Summary

In [ ]:
write_result = orchestrate_write(
    customer_summary_df,

    store="Gold",
    schema="demo",
    table_name="customer_summary",
    write_mode="overwrite",



    dq_diagnostics="sample",  # "none" keeps metrics only; "full" returns failed rows.
)
writes["customer_summary_warehouse"] = write_result
# Inspect the DataFrame submitted to this Write.
display(write_result["dataframe"].limit(10))

# Optional: inspect Data Quality diagnostics and caller-managed token mapping.
# display(write_result.get("dq_result"))
# display((write_result.get("sensitive_result") or {}).get("support_mapping"))

# Optional: read the persisted target after a successful persistent execution.
# from fabricops_kit import read_warehouse_query
# display(read_warehouse_query("SELECT TOP 10 * FROM demo.customer_summary", store="Gold"))


# 5. Finalize

Complete the run after every Read, transformation and Write. Dry Run keeps its summary in the session only. Ungoverned and Governed record the real outcome; Governed also releases its activation reservation. A stopped notebook remains incomplete.

In [ ]:
# 5. Finalize
pipeline_result = finalize_pipeline(pipeline)